In [2]:
import pandas as pd
import numpy as np
import joblib

from tensorflow.keras.models import load_model

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

I0000 00:00:1781085621.551216    3936 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781085642.198704    3936 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781085651.288228    3936 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
crop = pd.read_csv("/workspaces/farmer_crop_climate_mismatch_system/datasets/Crop_recommendation.csv")

crop.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:
X = crop.drop("label", axis=1)

y = crop["label"]

print(X.shape)
print(y.shape)

(2200, 7)
(2200,)


In [5]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", len(label_encoder.classes_))

Classes: 22


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [7]:
scaler = joblib.load(
    "/workspaces/farmer_crop_climate_mismatch_system/models/neural_network_scaler.pkl"
)

print("Scaler Loaded Successfully")

Scaler Loaded Successfully


In [8]:
X_test_scaled = scaler.transform(
    X_test
)

print(X_test_scaled.shape)

(440, 7)


In [9]:
model = load_model(
    "/workspaces/farmer_crop_climate_mismatch_system/models/neural_network_model.h5"
)

print("Model Loaded Successfully")

E0000 00:00:1781085759.537108    3936 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model Loaded Successfully


In [10]:
prediction_probs = model.predict(
    X_test_scaled,
    verbose=0
)

y_pred = np.argmax(
    prediction_probs,
    axis=1
)

In [ ]:
y = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Model Accuracy: {accuracy*100:.2f}%"
)

Model Accuracy: 99.55%


In [23]:
train_probs = model.predict(
    scaler.transform(X_train),
    verbose=0
)

train_pred = np.argmax(
    train_probs,
    axis=1
)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

print(f"Training Accuracy : {train_accuracy*100:.2f}%")
print(f"Testing Accuracy  : {accuracy*100:.2f}%")

gap = (train_accuracy - accuracy) * 100

print(f"Accuracy Gap      : {gap:.2f}%")

Training Accuracy : 99.20%
Testing Accuracy  : 99.55%
Accuracy Gap      : -0.34%


In [12]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00        20
           6       0.95      1.00      0.98        20
           7       1.00      1.00      1.00        20
           8       1.00      1.00      1.00        20
           9       0.95      1.00      0.98        20
          10       1.00      1.00      1.00        20
          11       1.00      0.95      0.97        20
          12       1.00      1.00      1.00        20
          13       1.00      1.00      1.00        20
          14       1.00      1.00      1.00        20
          15       1.00      1.00      1.00        20
          16       1.00      1.00      1.00        20
          17       1.00    

In [13]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1  0  0  0  0 19  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 20  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0 20  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0

In [14]:
sample = X.iloc[0]

print(sample)

print(
    "\nActual Crop:",
    y.iloc[0]
)

N               90.000000
P               42.000000
K               43.000000
temperature     20.879744
humidity        82.002744
ph               6.502985
rainfall       202.935536
Name: 0, dtype: float64

Actual Crop: rice


In [15]:
sample_scaled = scaler.transform(
    [sample]
)

prediction = model.predict(
    sample_scaled,
    verbose=0
)

predicted_crop = label_encoder.inverse_transform(
    [np.argmax(prediction)]
)[0]

print(
    "Predicted Crop:",
    predicted_crop
)

Predicted Crop: rice


/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [16]:
random_samples = crop.sample(
    10,
    random_state=42
)

correct = 0

for _, row in random_samples.iterrows():

    actual_crop = row["label"]

    features = row.drop("label")

    features_scaled = scaler.transform(
        [features]
    )

    prediction = model.predict(
        features_scaled,
        verbose=0
    )

    predicted_crop = label_encoder.inverse_transform(
        [np.argmax(prediction)]
    )[0]

    print(
        f"Actual: {actual_crop} | Predicted: {predicted_crop}"
    )

    if actual_crop == predicted_crop:
        correct += 1

/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Actual: muskmelon | Predicted: muskmelon
Actual: watermelon | Predicted: watermelon
Actual: papaya | Predicted: papaya
Actual: papaya | Predicted: papaya
Actual: apple | Predicted: apple
Actual: mango | Predicted: mango


/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Actual: apple | Predicted: apple
Actual: mothbeans | Predicted: mothbeans
Actual: mungbean | Predicted: mungbean


/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Actual: lentil | Predicted: lentil


In [24]:
print(
    "\nValidation Test"
)

print(
    f"Correct Predictions: {correct}/10"
)

print(
    f"Accuracy: {(correct/10)*100:.2f}%"
)


Validation Test
Correct Predictions: 10/10
Accuracy: 100.00%


In [25]:
sample = X.iloc[0]

sample_scaled = scaler.transform(
    [sample]
)

prediction = model.predict(
    sample_scaled,
    verbose=0
)

confidence = np.max(
    prediction
) * 100

print(
    f"Prediction Confidence: {confidence:.2f}%"
)

Prediction Confidence: 99.86%


/workspaces/farmer_crop_climate_mismatch_system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
print("=" * 60)
print("MODEL FIT ANALYSIS")
print("=" * 60)

print(f"Training Accuracy : {train_accuracy*100:.2f}%")
print(f"Testing Accuracy  : {accuracy*100:.2f}%")
print(f"Accuracy Gap      : {gap:.2f}%")

print("\nConclusion:")

if abs(gap) < 1:
    print("✅ Model is Well Fitted")
    print("✅ No Overfitting Detected")
    print("✅ No Underfitting Detected")
elif gap > 5:
    print("⚠ Overfitting Detected")
else:
    print("⚠ Further Investigation Required")

MODEL FIT ANALYSIS
Training Accuracy : 99.20%
Testing Accuracy  : 99.55%
Accuracy Gap      : -0.34%

Conclusion:
✅ Model is Well Fitted
✅ No Overfitting Detected
✅ No Underfitting Detected
